# Notebook 04 — Aggregation & Export

**Goal:** turn the fact table into the nine aggregate tables Power BI uses, check each one, and write a manifest.

**Input:** `data/interim/fact_validations.parquet` **Output:** `data/processed/*.csv`, `data/processed/_manifest.csv`

Rules applied to every table:

- `groupby().agg()` with **named aggregations**.
- **Boardings** = sum of `passenger_count` (net of reversed validations).
- **Load factor** is averaged **per trip** (one vehicle, one route, one date, one hour — see Notebook 03), not per validation row.
- Money is rounded to 2 decimals, rates and shares to 4.
- Key columns must have no nulls and the table must be unique on its grain — both are asserted before writing.
- Window operations via `transform`: `pct_of_daily_total`, `pct_of_monthly_boardings`, `pct_of_mode_total`, and month-over-month growth in the monthly and mode tables.
- Months are exported as `year_month` (`YYYY-MM`) plus `month_start` (ISO date `YYYY-MM-DD`) so Power BI can use a real date for slicers.
- The columns required by the brief come first, in the same order as before; extra helper columns (`month_start`, growth, `route_length_km`, `capacity`, `period_order`, `validations`) are added at the end so the existing Power BI queries keep working after a refresh.

In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

fact = pd.read_parquet(INTERIM_DIR / "fact_validations.parquet")
fact["day_type"] = fact["is_weekend"].map({False: "Weekday", True: "Weekend"})

# One row per trip, used for every load-factor average
trips = fact.drop_duplicates("trip_id")

TIME_PERIODS = ["Night (00-03)", "Early (04-06)", "AM Peak (07-09)",
                "Midday (10-15)", "PM Peak (16-18)", "Evening (19-23)"]

print(f"Fact rows: {len(fact):,} | trips: {len(trips):,} | months: {sorted(fact['year_month'].unique())}")

Fact rows: 79,149 | trips: 78,741 | months: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06']


## Export helper

`export` checks the table and writes it, then returns the information needed for the manifest.

In [2]:
manifest_rows = []


def month_start(year_month: pd.Series) -> pd.Series:
    return pd.to_datetime(year_month + "-01").dt.strftime("%Y-%m-%d")


def export(df: pd.DataFrame, name: str, keys: list[str]) -> pd.DataFrame:
    null_keys = df[keys].isna().sum()
    assert null_keys.sum() == 0, f"{name}: nulls in key columns\n{null_keys[null_keys > 0]}"
    assert not df.duplicated(subset=keys).any(), f"{name}: grain {keys} is not unique"

    path = PROCESSED_DIR / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8")
    manifest_rows.append({
        "file_name": path.name,
        "row_count": len(df),
        "column_count": df.shape[1],
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    })
    print(f"Saved {path.name}: {len(df):,} rows x {df.shape[1]} columns (keys checked: {keys})")
    return df

## 1. `agg_ridership_by_month`

- `unique_cards` leaves out `SINGLE_RIDE`, which is a placeholder for paper tickets and not a card.
- `avg_daily_boardings` = total boardings ÷ number of days in the month that have data.
- `boardings_mom_growth` = change versus the previous month (window: `pct_change`).

In [3]:
monthly = (
    fact.groupby("year_month", as_index=False)
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        active_days=("date", "nunique"),
    )
)
cards = (
    fact[fact["card_id"] != "SINGLE_RIDE"]
    .groupby("year_month", as_index=False)
    .agg(unique_cards=("card_id", "nunique"))
)
monthly = monthly.merge(cards, on="year_month", how="left", validate="one_to_one")

monthly["avg_daily_boardings"] = (monthly["total_boardings"] / monthly["active_days"]).round(2)
monthly = monthly.sort_values("year_month").reset_index(drop=True)
monthly["boardings_mom_growth"] = monthly["total_boardings"].transform("pct_change").round(4)
monthly["total_revenue"] = monthly["total_revenue"].round(2)
monthly["month_start"] = month_start(monthly["year_month"])

agg_ridership_by_month = export(
    monthly[["year_month", "total_boardings", "total_revenue", "unique_cards",
             "avg_daily_boardings", "month_start", "boardings_mom_growth"]],
    "agg_ridership_by_month", ["year_month"],
)
agg_ridership_by_month

Saved agg_ridership_by_month.csv: 6 rows x 7 columns (keys checked: ['year_month'])


,year_month,total_boardings,total_revenue,unique_cards,avg_daily_boardings,month_start,boardings_mom_growth
0,2024-01,14581,26798.67,9445,470.35,2024-01-01,NaN
1,2024-02,13607,25176.67,9038,469.21,2024-02-01,-0.0668
2,2024-03,14531,26589.69,9427,468.74,2024-03-01,0.0679
3,2024-04,13933,25620.73,9256,464.43,2024-04-01,-0.0412
4,2024-05,14671,26714.24,9511,473.26,2024-05-01,0.0530
5,2024-06,13924,25590.25,9153,464.13,2024-06-01,-0.0509


Six months, January–June 2024. Boardings move between about 13,600 and 14,700 per month. The dips in February and April are mostly calendar effects (fewer days): average daily boardings stay between about 460 and 475 all period. The first month has no previous month, so its growth value is empty by design.

## 2. `agg_ridership_by_hour`

- `avg_boardings_per_day` = boardings in that hour ÷ number of weekdays (or weekend days) in the data. Dividing by the number of *days* — not by the number of validation rows — is what makes this a per-day figure.
- `pct_of_daily_total` = share of that day type's boardings that happen in the hour (window: `transform("sum")`).

In [4]:
days_per_type = fact.groupby("day_type")["date"].nunique()

hourly = (
    fact.groupby(["hour", "day_type"], as_index=False)
    .agg(total_boardings=("passenger_count", "sum"))
)
hourly["avg_boardings_per_day"] = (
    hourly["total_boardings"] / hourly["day_type"].map(days_per_type)
).round(2)
hourly["pct_of_daily_total"] = (
    hourly["total_boardings"] / hourly.groupby("day_type")["total_boardings"].transform("sum")
).round(4)
hourly = hourly.sort_values(["day_type", "hour"]).reset_index(drop=True)

agg_ridership_by_hour = export(hourly, "agg_ridership_by_hour", ["hour", "day_type"])
print(days_per_type.to_string())
agg_ridership_by_hour.sort_values("avg_boardings_per_day", ascending=False).head(6)

Saved agg_ridership_by_hour.csv: 48 rows x 5 columns (keys checked: ['hour', 'day_type'])
day_type
Weekday    130
Weekend     52


,hour,day_type,total_boardings,avg_boardings_per_day,pct_of_daily_total
32,8,Weekend,3035,58.37,0.1244
17,17,Weekday,7488,57.60,0.1231
8,8,Weekday,7353,56.56,0.1209
41,17,Weekend,2918,56.12,0.1196
42,18,Weekend,2820,54.23,0.1156
18,18,Weekday,7004,53.88,0.1151


The data has 130 weekdays and 52 weekend days. The busiest hours are 08:00 and 17:00–18:00 on both day types, each with about 12% of that day type's boardings.

## 3. `agg_route_performance`

`district` is the route's own district from the routes table. `boardings_per_km` = boardings in the month ÷ route length.

In [5]:
route_keys = ["route_id", "route_name", "mode", "route_district", "year_month"]

route_perf = (
    fact.groupby(route_keys, as_index=False)
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        route_length_km=("route_length_km", "first"),
    )
)
route_load = (
    trips.groupby(["route_id", "year_month"], as_index=False)
    .agg(avg_load_factor=("load_factor", "mean"))
)
route_perf = route_perf.merge(route_load, on=["route_id", "year_month"], how="left", validate="one_to_one")

route_perf["boardings_per_km"] = (route_perf["total_boardings"] / route_perf["route_length_km"]).round(4)
route_perf["total_revenue"] = route_perf["total_revenue"].round(2)
route_perf["avg_load_factor"] = route_perf["avg_load_factor"].round(4)
route_perf = route_perf.rename(columns={"route_district": "district"})
route_perf["month_start"] = month_start(route_perf["year_month"])

route_perf = route_perf.sort_values(["year_month", "total_boardings"], ascending=[True, False])
agg_route_performance = export(
    route_perf[["route_id", "route_name", "mode", "district", "year_month",
                "total_boardings", "total_revenue", "boardings_per_km", "avg_load_factor",
                "month_start", "route_length_km"]].reset_index(drop=True),
    "agg_route_performance", ["route_id", "year_month"],
)
agg_route_performance.head()

Saved agg_route_performance.csv: 360 rows x 11 columns (keys checked: ['route_id', 'year_month'])


,route_id,route_name,mode,district,year_month,total_boardings,total_revenue,boardings_per_km,avg_load_factor,month_start,route_length_km
0,R032,Thomas Dam Line,Metro,Centro,2024-01,292,498.01,12.4255,0.0019,2024-01-01,23.5
1,R051,Daniel Lake Line,Bus,Centro,2024-01,285,521.42,58.1633,0.0143,2024-01-01,4.9
2,R039,Carlson Mountain Line,Bus,Centro,2024-01,280,501.23,66.6667,0.0142,2024-01-01,4.2
3,R031,Wilkerson Row Line,Tram,Centro,2024-01,278,516.28,16.4497,0.0055,2024-01-01,16.9
4,R021,Martin Knoll Line,Tram,Westend,2024-01,277,526.82,11.7373,0.0057,2024-01-01,23.6


60 routes × 6 months = 360 rows. `route_length_km` is kept in the table so the Power BI scatter can plot boardings against length.

## 4. `agg_mode_summary`

`boardings_mom_growth` is calculated within each mode (window: `groupby("mode").transform("pct_change")`).

In [6]:
mode_summary = (
    fact.groupby(["mode", "year_month"], as_index=False)
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        route_count=("route_id", "nunique"),
    )
    .sort_values(["mode", "year_month"])
)
mode_summary["boardings_per_route"] = (mode_summary["total_boardings"] / mode_summary["route_count"]).round(2)
mode_summary["boardings_mom_growth"] = (
    mode_summary.groupby("mode")["total_boardings"].transform("pct_change").round(4)
)
mode_summary["total_revenue"] = mode_summary["total_revenue"].round(2)
mode_summary["month_start"] = month_start(mode_summary["year_month"])

agg_mode_summary = export(
    mode_summary.sort_values(["year_month", "total_boardings"], ascending=[True, False]).reset_index(drop=True),
    "agg_mode_summary", ["mode", "year_month"],
)
agg_mode_summary.head(6)

Saved agg_mode_summary.csv: 18 rows x 8 columns (keys checked: ['mode', 'year_month'])


,mode,year_month,total_boardings,total_revenue,route_count,boardings_per_route,boardings_mom_growth,month_start
0,Bus,2024-01,8332,15363.63,34,245.06,NaN,2024-01-01
1,Tram,2024-01,4809,8823.31,20,240.45,NaN,2024-01-01
2,Metro,2024-01,1440,2611.73,6,240.00,NaN,2024-01-01
3,Bus,2024-02,7676,14213.12,34,225.76,-0.0787,2024-02-01
4,Tram,2024-02,4668,8575.81,20,233.40,-0.0293,2024-02-01
5,Metro,2024-02,1263,2387.74,6,210.50,-0.1229,2024-02-01


Every month, Bus carries the most passengers (34 routes), then Tram (20) and Metro (6), but boardings per route are almost the same for all three.

## 5. `agg_fare_type_mix`

In [7]:
fare_mix = (
    fact.groupby(["fare_type", "year_month"], as_index=False)
    .agg(boardings=("passenger_count", "sum"), revenue=("revenue", "sum"))
)
fare_mix["pct_of_monthly_boardings"] = (
    fare_mix["boardings"] / fare_mix.groupby("year_month")["boardings"].transform("sum")
).round(4)
fare_mix["revenue"] = fare_mix["revenue"].round(2)
fare_mix["month_start"] = month_start(fare_mix["year_month"])

fare_mix = fare_mix.sort_values(["year_month", "boardings"], ascending=[True, False]).reset_index(drop=True)
assert (fare_mix.groupby("year_month")["pct_of_monthly_boardings"].sum().round(2) == 1).all()

agg_fare_type_mix = export(fare_mix, "agg_fare_type_mix", ["fare_type", "year_month"])
agg_fare_type_mix.pivot(index="fare_type", columns="year_month", values="pct_of_monthly_boardings")

Saved agg_fare_type_mix.csv: 30 rows x 6 columns (keys checked: ['fare_type', 'year_month'])


year_month,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06
fare_type,,,,,,
Adult,0.5536,0.5611,0.5509,0.5591,0.5467,0.5570
Child,0.0780,0.0772,0.0766,0.0749,0.0815,0.0810
Pass Holder,0.0971,0.0951,0.1005,0.1020,0.1018,0.0993
Senior,0.1192,0.1169,0.1211,0.1173,0.1224,0.1145
Student,0.1521,0.1496,0.1508,0.1466,0.1477,0.1482


The fare mix is stable from month to month: Adult about 56%, Student 15%, Senior 12%, Pass Holder 10%, Child 8%. The shares add up to 100% in every month (asserted).

## 6. `agg_stop_activity`

The table starts from the **stops master**, so a stop with no boardings would still appear (with 0) instead of disappearing. `rank_within_district` uses `method="min"`: tied stops share a rank and the next rank is skipped (1, 2, 2, 4).

In [8]:
stop_boardings = fact.groupby("stop_id", as_index=False).agg(total_boardings=("passenger_count", "sum"))
stops_all = pd.read_parquet(INTERIM_DIR / "stops_clean.parquet").rename(columns={"district": "stop_district"})

stop_activity = (
    stops_all.merge(stop_boardings, on="stop_id", how="left", validate="one_to_one")
    .fillna({"total_boardings": 0})
    .rename(columns={"stop_district": "district"})
)
stop_activity["total_boardings"] = stop_activity["total_boardings"].astype("int64")
stop_activity["rank_within_district"] = (
    stop_activity.groupby("district")["total_boardings"].rank(method="min", ascending=False).astype("int64")
)
stop_activity = stop_activity.sort_values(["district", "rank_within_district", "stop_id"]).reset_index(drop=True)

agg_stop_activity = export(
    stop_activity[["stop_id", "stop_name", "district", "zone", "total_boardings",
                   "rank_within_district", "has_shelter"]],
    "agg_stop_activity", ["stop_id"],
)
print("Stops without any boarding:", (agg_stop_activity["total_boardings"] == 0).sum())
agg_stop_activity.head()

Saved agg_stop_activity.csv: 400 rows x 7 columns (keys checked: ['stop_id'])
Stops without any boarding: 0


,stop_id,stop_name,district,zone,total_boardings,rank_within_district,has_shelter
0,S0047,Joseph Coves Stop,Centro,Zone 3,237,1,False
1,S0314,Davis Locks Stop,Centro,Zone 2,233,2,True
2,S0394,Tucker Mills Stop,Centro,Zone 2,229,3,False
3,S0103,Jennifer Lights Stop,Centro,Zone 1,228,4,False
4,S0323,Brandon Camp Stop,Centro,Zone 1,228,4,False


All 400 stops are present. Stop totals sit in a narrow band, so a small difference in boardings can move a stop several ranks; the rank is useful for spotting the busiest stops in a district, not as a precise score.

## 7. `agg_vehicle_utilisation`

Like the stop table, this starts from the **vehicles master**, so an unused vehicle shows up with 0 boardings and 0 active days — those are exactly the vehicles the agency wants to find. `active_days` = number of distinct dates the vehicle recorded at least one validation.

In [9]:
vehicle_use = (
    fact.groupby("vehicle_id", as_index=False)
    .agg(total_boardings=("passenger_count", "sum"), active_days=("date", "nunique"))
)
vehicle_load = trips.groupby("vehicle_id", as_index=False).agg(avg_load_factor=("load_factor", "mean"))
vehicles_all = pd.read_parquet(INTERIM_DIR / "vehicles_clean.parquet")

vehicle_util = (
    vehicles_all[["vehicle_id", "vehicle_type", "depot", "vehicle_age_band", "capacity"]]
    .merge(vehicle_use, on="vehicle_id", how="left", validate="one_to_one")
    .merge(vehicle_load, on="vehicle_id", how="left", validate="one_to_one")
    .fillna({"total_boardings": 0, "active_days": 0, "avg_load_factor": 0})
)
vehicle_util = vehicle_util.astype({"total_boardings": "int64", "active_days": "int64"})
vehicle_util["avg_load_factor"] = vehicle_util["avg_load_factor"].round(4)
vehicle_util = vehicle_util.sort_values("total_boardings").reset_index(drop=True)

agg_vehicle_utilisation = export(
    vehicle_util[["vehicle_id", "vehicle_type", "depot", "vehicle_age_band",
                  "total_boardings", "avg_load_factor", "active_days", "capacity"]],
    "agg_vehicle_utilisation", ["vehicle_id"],
)
print("Vehicles with no boardings:", (agg_vehicle_utilisation["total_boardings"] == 0).sum())
agg_vehicle_utilisation.groupby("vehicle_type")[["total_boardings", "active_days", "avg_load_factor"]].mean().round(4)

Saved agg_vehicle_utilisation.csv: 180 rows x 8 columns (keys checked: ['vehicle_id'])
Vehicles with no boardings: 0


,total_boardings,active_days,avg_load_factor
vehicle_type,,,
Articulated Bus,394.7750,158.2250,0.0091
Metro Set,396.6667,158.5714,0.0018
Standard Bus,400.2469,158.2222,0.0158
Tram,755.4211,178.1579,0.0055


The table is sorted from least to most used, so the top rows are the candidates for "underutilised". Trams carry almost twice as many boardings per vehicle as any other type (about 755 against about 400) and run on more days. Metro sets carry no more per vehicle than a standard bus despite about nine times the capacity, so they have the lowest load factor.

## 8. `agg_peak_analysis`

`pct_of_mode_total` = share of a mode's boardings that happen in each time period (window: `transform("sum")` per mode). Rows are sorted in the natural order of the day.

In [10]:
peak = (
    fact.groupby(["time_period", "mode"], as_index=False)
    .agg(total_boardings=("passenger_count", "sum"))
)
peak_load = trips.groupby(["time_period", "mode"], as_index=False).agg(avg_load_factor=("load_factor", "mean"))
peak = peak.merge(peak_load, on=["time_period", "mode"], how="left", validate="one_to_one")

peak["avg_load_factor"] = peak["avg_load_factor"].round(4)
peak["pct_of_mode_total"] = (
    peak["total_boardings"] / peak.groupby("mode")["total_boardings"].transform("sum")
).round(4)
peak["period_order"] = peak["time_period"].map({p: i for i, p in enumerate(TIME_PERIODS, start=1)})
peak = peak.sort_values(["mode", "period_order"]).reset_index(drop=True)

agg_peak_analysis = export(
    peak[["time_period", "mode", "total_boardings", "avg_load_factor", "pct_of_mode_total", "period_order"]],
    "agg_peak_analysis", ["time_period", "mode"],
)
agg_peak_analysis.pivot(index="time_period", columns="mode", values="pct_of_mode_total").reindex(TIME_PERIODS)

Saved agg_peak_analysis.csv: 18 rows x 6 columns (keys checked: ['time_period', 'mode'])


mode,Bus,Metro,Tram
time_period,,,
Night (00-03),0.0088,0.0090,0.0085
Early (04-06),0.0342,0.0318,0.0355
AM Peak (07-09),0.2912,0.2950,0.2906
Midday (10-15),0.2479,0.2503,0.2479
PM Peak (16-18),0.3182,0.3145,0.3165
Evening (19-23),0.0997,0.0994,0.1010


`period_order` (1–6) lets Power BI sort the periods from night to evening instead of alphabetically. All three modes have the same daily profile: about 61% of boardings fall in the two peak periods.

## 9. `agg_transfer_behaviour`

`transfer_rate` = validations flagged as a transfer ÷ all validations in that month and mode.

In [11]:
transfers = (
    fact.groupby(["year_month", "mode"], as_index=False)
    .agg(transfer_count=("transfer_flag", "sum"), validations=("transfer_flag", "size"))
)
transfers["transfer_count"] = transfers["transfer_count"].astype("int64")
transfers["transfer_rate"] = (transfers["transfer_count"] / transfers["validations"]).round(4)
transfers["month_start"] = month_start(transfers["year_month"])
transfers = transfers.sort_values(["year_month", "mode"]).reset_index(drop=True)

agg_transfer_behaviour = export(
    transfers[["year_month", "mode", "transfer_count", "transfer_rate", "month_start", "validations"]],
    "agg_transfer_behaviour", ["year_month", "mode"],
)
agg_transfer_behaviour.pivot(index="year_month", columns="mode", values="transfer_rate")

Saved agg_transfer_behaviour.csv: 18 rows x 6 columns (keys checked: ['year_month', 'mode'])


mode,Bus,Metro,Tram
year_month,,,
2024-01,0.376,0.3749,0.3849
2024-02,0.37,0.35,0.3887
2024-03,0.3866,0.3811,0.3807
2024-04,0.3826,0.3505,0.3899
2024-05,0.3835,0.3802,0.3738
2024-06,0.3897,0.3753,0.3763


Between 35% and 39% of validations are transfers, for every mode and every month. There is no clear trend; the small dips (Metro in February and April) are within normal variation for the smaller Metro sample.

## 10. Manifest

In [12]:
manifest = pd.DataFrame(manifest_rows)
assert len(manifest) == 9, "Expected nine aggregate tables"
manifest.to_csv(PROCESSED_DIR / "_manifest.csv", index=False, encoding="utf-8")
manifest

,file_name,row_count,column_count,generated_at
0,agg_ridership_by_month.csv,6,7,2026-10-28T16:52:07
1,agg_ridership_by_hour.csv,48,5,2026-10-28T16:52:07
2,agg_route_performance.csv,360,11,2026-10-28T16:52:07
3,agg_mode_summary.csv,18,8,2026-10-28T16:52:07
4,agg_fare_type_mix.csv,30,6,2026-10-28T16:52:07
5,agg_stop_activity.csv,400,7,2026-10-28T16:52:07
6,agg_vehicle_utilisation.csv,180,8,2026-10-28T16:52:07
7,agg_peak_analysis.csv,18,6,2026-10-28T16:52:07
8,agg_transfer_behaviour.csv,18,6,2026-10-28T16:52:07


In [13]:
for row in manifest.itertuples():
    check = pd.read_csv(PROCESSED_DIR / row.file_name)
    assert check.shape == (row.row_count, row.column_count), row.file_name
print("All nine files read back with the row and column counts listed in the manifest.")

All nine files read back with the row and column counts listed in the manifest.


**Next:** load these tables into Power BI (`report.pbix`) or run the bonus `05_database_bonus.ipynb`.